# PCA of Returns: Statistical Factors

07/03 chose the factors in advance: market, size, value, momentum. Principal component analysis (PCA) lets the data choose
them. The eigenvectors of the covariance matrix are portfolios whose returns are uncorrelated, ordered by how much variance
they explain. For equities the first one is almost always "the market". For interest rates the first three are the level,
slope and curvature of the yield curve, and they explain nearly all of its movements. This notebook computes PCA with the
spectral decomposition and the SVD, separates real factors from noise, and uses the components for risk models and hedging.

## 0. Math link

- [Spectral decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/055_spectral_decomposition.ipynb):
  a symmetric matrix is $V\Lambda V^\top$ with orthonormal eigenvectors.
- [Singular value decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/057_singular_value_decomposition.ipynb):
  the SVD of the centred data matrix gives the same components without forming the covariance matrix.
- Factor models and the factor-model covariance matrix from [07/03](./03_multi_factor_models_and_return_attribution.ipynb);
  noisy eigenvalues of a large sample covariance matrix from [06/04](../06_estimation_error/04_means_are_hard_covariances_are_easier.ipynb).

## 1. Introduction

A risk manager with 50 stocks and a bond desk with 8 points on the yield curve ask the same question: how many independent
sources of risk are really there? PCA answers by rotating the returns into uncorrelated components and ranking them by
variance. A few components usually carry most of the risk, and the rest look like noise.

After this notebook you can:

- compute principal components from the covariance or correlation matrix, and via the SVD;
- read explained-variance shares and component loadings (eigen-portfolios);
- compare statistical factors with known economic factors;
- use the Marchenko–Pastur bound to tell factor eigenvalues from noise;
- build a statistical factor model of the covariance matrix;
- interpret the level, slope and curvature components of the yield curve and hedge against them.

## 2. Theory

### Principal components

Let $\Sigma$ be the $N\times N$ covariance matrix of returns $r$. Its spectral decomposition is

$$
\Sigma = V\Lambda V^\top = \sum_{k=1}^N \lambda_k v_k v_k^\top, \qquad \lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_N \ge 0, \quad V^\top V = I
$$

The $k$-th principal component is $p_k = v_k^\top(r - \mu)$. Then

$$
\operatorname{Var}(p_k) = v_k^\top\Sigma v_k = \lambda_k, \qquad \operatorname{Cov}(p_j, p_k) = 0 \ (j \ne k), \qquad \sum_k \lambda_k = \operatorname{tr}\Sigma = \sum_i \sigma_i^2
$$

The share of total variance explained by component $k$ is $\lambda_k/\sum_j\lambda_j$. $v_1$ is the direction of maximum variance among unit
vectors: it maximizes $v^\top\Sigma v$ subject to $\lVert v\rVert = 1$ (Lagrange: $\Sigma v = \lambda v$).

Each $v_k$ can be read as a portfolio, an **eigen-portfolio**; its sign is arbitrary.

### PCA via the SVD

With the centred data matrix $X_c \in \mathbb{R}^{n\times N}$ (rows = months), $\hat\Sigma = X_c^\top X_c/(n-1)$. If $X_c = USV^\top$, then

$$
\hat\Sigma = V\frac{S^2}{n-1}V^\top
$$

so the right singular vectors are the eigenvectors and $\hat\lambda_k = s_k^2/(n-1)$. The component time series are $X_cV = US$. This avoids forming
$X_c^\top X_c$ (07/02: it squares the condition number).

### Covariance or correlation?

PCA on the covariance matrix lets volatile assets dominate. PCA on the correlation matrix (standardized returns) treats all
assets equally. For a stock universe the correlation matrix is common; for yield changes, which are in the same units, the
covariance matrix.

### Equicorrelation

$N$ assets with variance $\sigma^2$ and common correlation $\rho$: $\lambda_1 = \sigma^2(1 + (N-1)\rho)$ with $v_1 = \mathbf 1/\sqrt N$ (the equal-weighted
market), and $\lambda_2 = \dots = \lambda_N = \sigma^2(1 - \rho)$. The market component explains $(1 + (N-1)\rho)/N \to \rho$ of the variance.

### Signal or noise? Marchenko–Pastur

Even when the true correlation matrix is the identity, the eigenvalues of a sample correlation matrix from $n$ observations of
$N$ assets spread out. As $N, n \to \infty$ with $q = N/n$ fixed, they fill the interval

$$
\lambda_\pm = \big(1 \pm \sqrt{q}\big)^2
$$

Sample eigenvalues well above $\lambda_+$ indicate real factors; those inside the interval are indistinguishable from noise.

### Statistical factor model

Keep the first $K$ components:

$$
\Sigma \approx V_K\Lambda_K V_K^\top + D
$$

with $D$ the diagonal of the residual covariance. It has the structure of 07/03 with the factors estimated rather than
specified. The components are not unique economic factors: any rotation of the first $K$ eigenvectors spans the same
subspace.

### The yield curve

Monthly changes in yields of different maturities are highly correlated. PCA of their covariance matrix typically
gives (Litterman and Scheinkman 1991):

- **level**: all maturities move together, typically 70–90% of the variance;
- **slope**: short and long ends move in opposite directions;
- **curvature**: the middle moves against both ends.

Together they explain about 98%. A bond portfolio hedged only for duration is hedged against the level component; hedging
slope and curvature needs more instruments.

## 3. Worked Examples

### Example 1: two assets

Equal volatility $\sigma$, correlation $\rho$: $\Sigma = \sigma^2\begin{pmatrix}1 & \rho\\ \rho & 1\end{pmatrix}$, eigenvalues $\sigma^2(1 \pm \rho)$, eigenvectors
$(1, 1)/\sqrt2$ (the sum) and $(1, -1)/\sqrt2$ (the spread). With $\rho = 0.8$ the sum explains $1.8/2 = 90\%$ of the variance.

### Example 2: fifty stocks

Equicorrelation $\rho = 0.3$, $N = 50$: the market component explains $(1 + 49\cdot 0.3)/50 = 15.7/50 = 31.4\%$; each of the other 49
explains $0.7/50 = 1.4\%$.

### Example 3: the noise band

$N = 50$ assets, $n = 120$ months: $q = 0.417$, $\sqrt q = 0.645$, $\lambda_+ = 1.645^2 = 2.71$, $\lambda_- = 0.355^2 = 0.126$. A sample correlation
eigenvalue of 2.0 is consistent with pure noise; one of 6 is not.

### Example 4: hedging the level

A portfolio's yield exposure per maturity is its DV01 vector $d$ (CHF per basis point). Its exposure to component $k$ is $d^\top v_k$ per unit of
the component. A barbell (2-year and 30-year) and a bullet (10-year) with the same total DV01 have the same level exposure if
$v_1$ is flat, but different slope and curvature exposures.

## 4. Finance Application

### Where it matters

- **Risk models.** Statistical factor models complement fundamental ones (07/03); they pick up structure no one named in
  advance, such as a crowded trade.
- **Diversification diagnostics.** The share of the first component measures how much a portfolio is "one bet". In crises
  equity correlations rise and the first component's share jumps.
- **Fixed income.** Yield-curve risk is reported and hedged in level, slope and curvature terms; scenario analysis shifts
  the components.
- **Noise filtering.** Replacing eigenvalues inside the Marchenko–Pastur band by their average ("clipping") cleans a
  covariance matrix before optimization (08/02).

### The usual misreadings

- "PC2 is the value factor." Components beyond the first are statistical rotations; they may resemble economic factors
  but are not identified as such, and they change from sample to sample (exercise 6).
- "The first component explains 40%, so the other 60% is diversifiable." The other components can still be systematic;
  only the noise components are.
- "PCA on prices." PCA needs (approximately) stationary data: returns or yield changes, not price levels.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

factors = ["MKT", "SMB", "HML", "MOM"]
f_mu = np.array([0.06, 0.02, 0.03, 0.05])
f_vol = np.array([0.16, 0.08, 0.09, 0.12])
f_corr = np.array([[1.00, 0.20, -0.10, -0.15],
                   [0.20, 1.00, 0.05, 0.00],
                   [-0.10, 0.05, 1.00, -0.30],
                   [-0.15, 0.00, -0.30, 1.00]])
Sigma_f = np.outer(f_vol, f_vol) * f_corr

### Fifty stocks driven by four factors

The same setup as 07/03: 50 stocks, four factors, 20% idiosyncratic volatility, ten years of monthly excess returns.

In [ ]:
n, N = 120, 50
F = f_mu / 12 + rng.standard_normal((n, 4)) @ np.linalg.cholesky(Sigma_f / 12).T
B_true = np.column_stack([1 + 0.25 * rng.standard_normal(N), 0.2 + 0.4 * rng.standard_normal(N),
                          0.1 + 0.4 * rng.standard_normal(N), 0.2 * rng.standard_normal(N)]).T
idio = np.full(N, 0.20)
Y = F @ B_true + idio / np.sqrt(12) * rng.standard_normal((n, N))
Sigma_true = B_true.T @ Sigma_f @ B_true + np.diag(idio**2)

Yc = Y - Y.mean(axis=0)
S_hat = Yc.T @ Yc / (n - 1) * 12                       # annualized sample covariance

lam, V = np.linalg.eigh(S_hat)
lam, V = lam[::-1], V[:, ::-1]                         # descending order

U, s, Vt = np.linalg.svd(Yc, full_matrices=False)
lam_svd = s**2 / (n - 1) * 12
print("eigenvalues equal (eigh vs SVD):", np.allclose(lam, lam_svd))
print("eigenvectors equal up to sign:", np.allclose(np.abs(V[:, :5]), np.abs(Vt.T[:, :5])))

share = lam / lam.sum()
print("explained variance, first 6 components:", (share[:6] * 100).round(1), "%")
print(f"cumulative, first 4: {share[:4].sum():.1%}")

The eigendecomposition and the SVD agree. The first component explains 41% of the variance; the next ones only 2–4% each,
and the first four together 51%. Most of each stock's variance is idiosyncratic (20% volatility against about 16% from the
market), so the many small components are mostly noise.

### Do the components match the factors?

Correlate the first component's returns with the market factor, and regress each true factor on the first four components.

In [ ]:
V = V * np.sign(V.sum(axis=0))                         # fix sign: positive sum of loadings
P = Yc @ V[:, :4]                                      # component returns (monthly)
print(f"corr(PC1, MKT) = {np.corrcoef(P[:, 0], F[:, 0])[0, 1]:.3f}")
print(f"corr(PC1 loadings, true MKT betas) = {np.corrcoef(V[:, 0], B_true[0])[0, 1]:.3f}")
print(f"PC1 eigen-portfolio weights: min {V[:, 0].min():.3f}, max {V[:, 0].max():.3f}, equal weight would be {1 / np.sqrt(N):.3f}")

Xp = np.column_stack([np.ones(n), P])
r2 = {}
for k, name in enumerate(factors):
    b = np.linalg.lstsq(Xp, F[:, k], rcond=None)[0]
    e = F[:, k] - Xp @ b
    r2[name] = 1 - e @ e / ((F[:, k] - F[:, k].mean()) @ (F[:, k] - F[:, k].mean()))
print("R^2 of each true factor on the first 4 components:", {k: round(v, 3) for k, v in r2.items()})

PC1 is the market: its returns have correlation 0.98 with MKT, and its loadings have correlation 0.91 with the true market
betas. Its eigen-portfolio is long every stock, with weights between 0.02 and 0.21. The other three factors are recovered
poorly: the first four components explain 96% of the variance of MKT but only 12% of SMB, 43% of HML and 33% of MOM. Their
contribution to each stock's variance is small relative to the idiosyncratic noise, so with 120 months PCA cannot isolate them.

### Signal or noise: the correlation matrix and Marchenko–Pastur

In [ ]:
d = np.sqrt(np.diag(S_hat))
C_hat = S_hat / np.outer(d, d)
lam_c = np.sort(np.linalg.eigvalsh(C_hat))[::-1]
q = N / n
lam_plus, lam_minus = (1 + np.sqrt(q)) ** 2, (1 - np.sqrt(q)) ** 2
print(f"Marchenko-Pastur band: [{lam_minus:.3f}, {lam_plus:.3f}]")
print("largest correlation eigenvalues:", lam_c[:6].round(2))
print(f"eigenvalues above the band: {np.sum(lam_c > lam_plus)}")

Z = rng.standard_normal((n, N))                        # pure noise, true correlation = identity
lam_noise = np.sort(np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False)))[::-1]
print(f"pure noise: largest {lam_noise[0]:.2f}, smallest {lam_noise[-1]:.3f}")

Only one eigenvalue (19.3) lies above the Marchenko–Pastur edge of 2.71; the second largest, 1.93, is below the largest
eigenvalue of pure noise (2.51). Statistically, this universe has one detectable factor in ten years of data, even though four
are present.

### A statistical factor model for the covariance matrix

Estimate from 60 months, keep $K$ components plus a diagonal residual, and compare the true volatility of the minimum-variance
portfolio with the sample covariance and with the known-factor model of 07/03. 200 repetitions.

In [ ]:
def gmv(S):
    w = np.linalg.solve(S, np.ones(S.shape[0]))
    return w / w.sum()

def true_vol(w):
    return np.sqrt(w @ Sigma_true @ w)

def pca_cov(S, K):
    lam_s, V_s = np.linalg.eigh(S)
    lam_s, V_s = lam_s[::-1], V_s[:, ::-1]
    S_K = V_s[:, :K] @ np.diag(lam_s[:K]) @ V_s[:, :K].T
    return S_K + np.diag(np.diag(S - S_K))

Ks = [1, 2, 4, 8, 15]
out = {f"PCA K={K}": [] for K in Ks}
out["sample"], out["known factors"] = [], []
for rep in range(200):
    Fs = f_mu / 12 + rng.standard_normal((60, 4)) @ np.linalg.cholesky(Sigma_f / 12).T
    Ys = Fs @ B_true + idio / np.sqrt(12) * rng.standard_normal((60, N))
    S = np.cov(Ys, rowvar=False) * 12
    out["sample"].append(true_vol(gmv(S)))
    for K in Ks:
        out[f"PCA K={K}"].append(true_vol(gmv(pca_cov(S, K))))
    Xs = np.column_stack([np.ones(60), Fs])
    Bs = np.linalg.pinv(Xs) @ Ys
    Es = Ys - Xs @ Bs
    out["known factors"].append(true_vol(gmv(Bs[1:].T @ (np.cov(Fs, rowvar=False) * 12) @ Bs[1:]
                                             + np.diag(Es.var(axis=0, ddof=5) * 12))))
gmv_table = pd.Series({k: np.mean(v) for k, v in out.items()})
gmv_opt = true_vol(gmv(Sigma_true))
print(f"true optimum {gmv_opt:.2%}")
gmv_table.map(lambda v: f"{v:.2%}")

Any PCA model with $K$ between 1 and 4 brings the minimum-variance portfolio to about 10.8%, as good as the known-factor
model of 07/03 and far better than the sample covariance (23.8%). Keeping too many components (15) lets noise back in.

### The yield curve

Monthly changes of Swiss franc swap yields at eight maturities (basis points), simulated from level, slope and curvature
shocks with Nelson–Siegel shapes plus small independent noise. The shapes and volatilities are stylized.

In [ ]:
mats = np.array([1, 2, 3, 5, 7, 10, 20, 30])
tau = 3.0
ns_slope = (1 - np.exp(-mats / tau)) / (mats / tau)               # Nelson-Siegel: 1 at the short end, toward 0 at the long end
ns_curv = ns_slope - np.exp(-mats / tau)                          # hump in the middle
slope_shape, curv_shape = ns_slope - ns_slope.mean(), ns_curv - ns_curv.mean()   # centred: shocks that tilt and bend the curve
shapes = np.column_stack([np.ones(mats.size), slope_shape, curv_shape])
shock_vol = np.array([10.0, 20.0, 20.0])                          # bp per month
n_y = 240
shocks = rng.standard_normal((n_y, 3)) * shock_vol
dY = shocks @ shapes.T + 2.0 * rng.standard_normal((n_y, mats.size))   # bp changes

C_y = np.cov(dY, rowvar=False)
lam_y, V_y = np.linalg.eigh(C_y)
lam_y, V_y = lam_y[::-1], V_y[:, ::-1]
V_y = V_y * np.sign(V_y[-1])                                      # sign convention: positive at 30 years
share_y = lam_y / lam_y.sum()
print("explained variance, first 4:", (share_y[:4] * 100).round(1), "%,  cumulative 3:", f"{share_y[:3].sum():.1%}")
print("PC1 loadings:", V_y[:, 0].round(2))
print("PC2 loadings:", V_y[:, 1].round(2))
print("PC3 loadings:", V_y[:, 2].round(2))

Three components explain 98.3% of the variance: level 77.9% (loadings flat at about 0.35), slope 18.8% (from −0.56 at one year to
+0.49 at thirty) and curvature 1.6% (positive at both ends, negative in the 5–10-year range).

### Hedging a barbell against a bullet

Two portfolios with the same total DV01 of CHF 10,000 per basis point: a bullet in the 10-year and a 50/50 barbell in the
2-year and 30-year. Their exposures to the first three components (CHF per unit of each component, i.e. per one-standard-deviation
move scaled by $\sqrt{\lambda_k}$):

In [ ]:
dv01_bullet = np.zeros(mats.size); dv01_bullet[mats == 10] = 10_000
dv01_barbell = np.zeros(mats.size); dv01_barbell[mats == 2] = 5_000; dv01_barbell[mats == 30] = 5_000
exp_table = pd.DataFrame({
    "bullet 10y": [dv01_bullet @ V_y[:, k] * np.sqrt(lam_y[k]) for k in range(3)],
    "barbell 2y/30y": [dv01_barbell @ V_y[:, k] * np.sqrt(lam_y[k]) for k in range(3)],
}, index=["level (1 sd)", "slope (1 sd)", "curvature (1 sd)"])
exp_table["difference"] = exp_table["barbell 2y/30y"] - exp_table["bullet 10y"]
exp_table.round(0)

With equal total DV01 and a flat level component, the two portfolios have the same level exposure (about CHF 104,000 for a
one-standard-deviation level move). They differ in slope (the bullet has CHF 24,000 more) and curvature (the barbell has
CHF 20,000 more): a duration-matched switch from bullet to barbell is a bet on the shape of the curve.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
k = np.arange(1, N + 1)
ax.bar(k, lam_c, color="tab:blue", label="sample correlation eigenvalues (factor model)")
ax.plot(k, lam_noise, "k.", markersize=4, label="pure noise, same n and N")
ax.axhline(lam_plus, color="tab:red", linestyle="--", label=f"Marchenko-Pastur upper edge {lam_plus:.2f}")
ax.set_yscale("log")
ax.set_title("Eigenvalues of the 50 × 50 correlation matrix, 120 months")
ax.set_xlabel("component")
ax.set_ylabel("eigenvalue (log scale)")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(B_true[0], V[:, 0], s=15)
ax.set_title(f"PC1 loadings vs true market betas (corr {np.corrcoef(V[:, 0], B_true[0])[0, 1]:.2f})")
ax.set_xlabel("true market beta")
ax.set_ylabel("PC1 loading (eigenvector entry)")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for kk, name in enumerate(["level", "slope", "curvature"]):
    ax.plot(mats, V_y[:, kk], "o-", label=f"PC{kk + 1} ({name}), {share_y[kk]:.1%}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xscale("log")
ax.set_xticks(mats)
ax.set_xticklabels(mats)
ax.set_title("Principal components of monthly yield changes")
ax.set_xlabel("maturity (years, log scale)")
ax.set_ylabel("loading")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(gmv_table.index, gmv_table.values, color=["tab:blue"] * len(Ks) + ["grey", "tab:green"])
ax.axhline(gmv_opt, color="black", linewidth=1, label="true optimum")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("True volatility of the minimum-variance portfolio (50 stocks, 60 months, average of 200)")
ax.set_ylabel("true annual volatility")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Two assets with equal volatility and correlation 0.6. Compute the eigenvalues (as multiples of $\sigma^2$), eigenvectors, and the share of
   the first component.
2. Twenty stocks with common correlation 0.5. What share of the variance does the market component explain? What is the
   limit for many stocks?
3. A risk report says "PC1 explains 45% of the variance of our equity book, up from 25% a year ago." **Interpret** this for the
   portfolio's diversification and for a hedge with index futures.

### Intermediate

4. Show numerically that the component returns $X_cV$ are uncorrelated and have variances $\hat\lambda_k$ (monthly). Check that the total
   variance is preserved.
5. Hedge the barbell so that its level **and** slope exposures equal those of the bullet, using the 5-year and 20-year
   points as additional instruments (solve a 2×2 system for their DV01s). What curvature exposure is left?

### Advanced

6. Stability: estimate the eigenvectors of the stock covariance matrix on months 1–60 and 61–120 separately. Compute the absolute
   inner product $|v_k^{(1)\top}v_k^{(2)}|$ for $k = 1, \dots, 5$. Which components are stable?
7. Eigenvalue clipping: replace the correlation eigenvalues below $\lambda_+$ by their average (keeping the trace), rebuild the
   covariance matrix, and compare the true volatility of its minimum-variance portfolio with the sample and PCA versions over
   the same 200 repetitions with 60 months.

## 8. Solutions / Checks

In [ ]:
# 1.
lam1 = np.linalg.eigvalsh(np.array([[1, 0.6], [0.6, 1]]))[::-1]
print(f"1. eigenvalues {lam1}, share of PC1 {lam1[0] / lam1.sum():.0%}")

# 2.
print(f"2. share {(1 + 19 * 0.5) / 20:.1%}, limit 50%")

# 4.
Pm = Yc @ V
Cp = np.cov(Pm, rowvar=False)
print(f"4. max off-diagonal |cov| {np.abs(Cp - np.diag(np.diag(Cp))).max():.1e}; "
      f"variances = lambda/12: {np.allclose(np.diag(Cp), lam / 12)}; total variance preserved: "
      f"{np.isclose(np.trace(Cp), np.trace(np.cov(Yc, rowvar=False)))}")

In [ ]:
# 5.
E = np.column_stack([V_y[:, 0], V_y[:, 1]])                  # level and slope loadings
target = (dv01_bullet - dv01_barbell) @ E                     # exposure gap to close
inst = np.zeros((mats.size, 2)); inst[mats == 5, 0] = 1; inst[mats == 20, 1] = 1
x = np.linalg.solve(E.T @ inst, target)
hedged = dv01_barbell + inst @ x
print(f"5. add DV01 {x[0]:,.0f} at 5y and {x[1]:,.0f} at 20y (CHF per bp)")
for k, name in enumerate(["level", "slope", "curvature"]):
    print(f"   {name:9s}: bullet {dv01_bullet @ V_y[:, k]:8.0f}, hedged barbell {hedged @ V_y[:, k]:8.0f}")

# 6.
def eigvecs(Ysub):
    l, v = np.linalg.eigh(np.cov(Ysub, rowvar=False))
    return v[:, ::-1]
V1, V2 = eigvecs(Y[:60]), eigvecs(Y[60:])
print("6. |v1'v2| for k = 1..5:", np.abs(np.sum(V1[:, :5] * V2[:, :5], axis=0)).round(2))

In [ ]:
# 7.
def clipped_cov(S, n_obs):
    dd = np.sqrt(np.diag(S))
    C = S / np.outer(dd, dd)
    l, v = np.linalg.eigh(C)
    lp = (1 + np.sqrt(S.shape[0] / n_obs)) ** 2
    noise = l < lp
    l_new = l.copy()
    l_new[noise] = l[noise].mean()
    C_new = v @ np.diag(l_new) @ v.T
    dn = np.sqrt(np.diag(C_new))
    C_new = C_new / np.outer(dn, dn)
    return C_new * np.outer(dd, dd)

r7 = {"sample": [], "PCA K=4": [], "clipped": []}
for rep in range(200):
    Fs = f_mu / 12 + rng.standard_normal((60, 4)) @ np.linalg.cholesky(Sigma_f / 12).T
    Ys = Fs @ B_true + idio / np.sqrt(12) * rng.standard_normal((60, N))
    S = np.cov(Ys, rowvar=False) * 12
    r7["sample"].append(true_vol(gmv(S)))
    r7["PCA K=4"].append(true_vol(gmv(pca_cov(S, 4))))
    r7["clipped"].append(true_vol(gmv(clipped_cov(S, 60))))
print("7.", {k: f"{np.mean(v):.2%}" for k, v in r7.items()}, f"optimum {gmv_opt:.2%}")

**1.** Eigenvalues $1.6\sigma^2$ and $0.4\sigma^2$, eigenvectors $(1,1)/\sqrt2$ and $(1,-1)/\sqrt2$; the first explains 80%.

**2.** $(1 + 19\cdot 0.5)/20 = 52.5\%$; for many stocks it tends to $\rho = 50\%$.

**3.** The book has become much more of a single bet: a larger share of its variance moves with one common factor, most
likely the market. Diversification across names is helping less. On the positive side, an index-future hedge now removes
a larger share of the risk; on the negative side, if correlations rose because of market stress, the risk not explained by
PC1 may also have become more correlated with it in a crash.

**4.** The component returns are uncorrelated up to rounding, their variances are the eigenvalues (here divided by 12 because
the eigenvalues were annualized), and their total variance equals the total variance of the stocks.

**5.** Selling CHF 3,678 of DV01 at 5 years and buying CHF 3,552 at 20 years matches the bullet's level and slope exposures exactly.
The curvature exposures still differ (−2,179 for the bullet against 5,062 for the hedged barbell, per unit of the component):
two instruments match two exposures, and the third remains a bet on the shape of the curve.

**6.** The overlaps are 0.97, 0.11, 0.10, 0.01 and 0.26. The first eigenvector is very stable across the two halves; from the second on
the overlap is close to what random directions would give. The
leading component is a robust feature of the data, the others are partly sample-specific rotations.

**7.** Clipping gives 10.85% against 23.6% for the sample covariance and 10.9% for PCA with $K = 4$ (optimum 9.5%): it matches the
statistical factor model without choosing $K$ by hand.

## 9. Conclusion

- PCA rotates returns into uncorrelated components via $\Sigma = V\Lambda V^\top$; $\lambda_k/\sum\lambda$ is the share of variance of component $k$.
- The SVD of the centred data gives the same components without forming the covariance matrix.
- For equities the first component is the market, with loadings close to market betas; later components are rotations of
  the remaining factors and are less stable.
- Eigenvalues above the Marchenko–Pastur edge $(1 + \sqrt{N/n})^2$ are signal; the rest are consistent with noise.
- A statistical factor model $V_K\Lambda_KV_K^\top + D$ fixes the instability of a large sample covariance matrix.
- Yield-curve changes reduce to level, slope and curvature; hedging duration covers only the first.

This closes module 07. Next: [Module 08, 01 Why mean-variance optimization breaks](../08_robust_portfolio_construction/01_why_mean_variance_optimization_breaks.ipynb).